# 41 Fresh ScienceWorld restricted-action validation
CPU runtime. Up to eight new development cases (variations 1 and 2). No GPU, model or token required. This is a feasibility gate before expanding validation, not a final labelled dataset.

Search horizons: 1, 2, 3 further calls. Labels are provisional and relative to a restricted action set and projected state. Scope coverage, including hidden quantities, still needs review. The initial parent subtrees are included conservatively in this run; this can include a full room. The sewer is always included. Original runs remain unchanged.

Default budgets: 90 seconds between-call search budget per case, 512 nodes, 24 permitted actions per node. If the action limit is exceeded, the case stops unresolved; no top-k pruning. Gold-path discovery, initial replay and slow Java calls can add time beyond these budgets. Results are saved after each case. A cutoff is unresolved, not irreversible.


In [ ]:
import sys, subprocess, os
os.environ['JAVA_TOOL_OPTIONS'] = '-Xlog:os+container=off'
subprocess.run(['java', '-version'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install',
 'git+https://github.com/allenai/ScienceWorld.git@e8216d6044e8e39be9fcb185e3b2dfb602584b52'], check=True)
print('Pinned ScienceWorld installed. No model packages needed.')


In [ ]:
from google.colab import drive
from pathlib import Path
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/ADNN_Irreversibility')
assert PROJECT.is_dir(), 'Project folder not found on Drive.'
print('Project:', PROJECT)


In [ ]:
from pathlib import Path
Path("/content/40_scienceworld_drainage_recheck_colab.py").write_text('"""Offline drainage-aware projection recheck. Original results preserved; no final labels."""\nimport gzip, json, hashlib, math\nfrom pathlib import Path\nfrom datetime import datetime, timezone\n\nROOT=Path(\'/content/drive/MyDrive/ADNN_Irreversibility/results\')\nPILOT=ROOT/\'scienceworld_physical_pilot_20261107T091929052223Z\'\nRECOVERY=ROOT/\'scienceworld_recovery_diagnostic_20261107T094243161617Z\'\n\ndef index(tree):\n    nodes={}; parents={}\n    def visit(node,parent=None):\n        if not isinstance(node,dict) or \'uuid\' not in node:raise ValueError(\'Unsupported object schema\')\n        uid=str(node[\'uuid\'])\n        if uid in nodes:raise ValueError(\'Duplicate UUID\')\n        nodes[uid]=node;parents[uid]=parent\n        children=node.get(\'contents\',{})\n        if not isinstance(children,dict):raise ValueError(\'Unsupported contents schema\')\n        for c in children.values():visit(c,uid)\n    visit(tree);return nodes,parents\n\ndef descendants(nodes,roots):\n    result=set()\n    def visit(uid):\n        if uid in result or uid not in nodes:return\n        result.add(uid)\n        for c in nodes[uid].get(\'contents\',{}).values():visit(str(c[\'uuid\']))\n    for uid in roots:visit(uid)\n    return result\n\ndef make_scope(before,ids):\n    nodes,parents=index(before)\n    if not ids or any(i not in nodes for i in ids):raise ValueError(\'Binding objects absent\')\n    # Include the drainage destination in every case, not only those that failed.\n    # Resolve by unique name in the initial state; keep its UUID thereafter.\n    sewers=[uid for uid,node in nodes.items() if node.get(\'name\')==\'sewer\']\n    if len(sewers)!=1:raise ValueError(\'Expected one sewer; drainage scope cannot be established\')\n    ids=set(ids)|set(sewers)\n    core=descendants(nodes,ids)\n    context={parents[i] for i in core if parents[i] is not None}-core\n    return {\'roots\':sorted(ids),\'core\':sorted(core),\'context\':sorted(context),\'drainage_roots\':sewers,\'scope_version\':\'v2_drainage\'}\n\ndef project(tree,scope):\n    nodes,parents=index(tree)\n    ids=set(scope[\'core\'])|set(scope[\'context\'])|descendants(nodes,scope[\'roots\'])\n    result={}\n    for uid in sorted(ids):\n        if uid not in nodes:result[uid]={\'missing\':True};continue\n        n=nodes[uid]\n        result[uid]={\'parent_uuid\':parents[uid],\n          \'properties\':{k:v for k,v in n.items() if k!=\'contents\'},\n          \'children\':sorted(str(c[\'uuid\']) for c in n.get(\'contents\',{}).values())}\n    return result\n\ndef differences(a,b,path=()):\n    if type(a)!=type(b):return [(path,a,b)]\n    if isinstance(a,dict):\n        result=[]\n        for k in sorted(a.keys()|b.keys()):\n            if k not in a or k not in b:result.append((path+(k,),a.get(k),b.get(k)))\n            else:result.extend(differences(a[k],b[k],path+(k,)))\n        return result\n    return [] if a==b else [(path,a,b)]\n\ndef compare(a,b):\n    ds=differences(a,b)\n    def thermal(d):\n        path,x,y=d\n        return path[-2:]==(\'propMaterial\',\'temperatureC\') and all(type(v) in (int,float) and math.isfinite(v) for v in (x,y))\n    return {\'exact_projection_match\':not ds,\'thermal_only_residual\':bool(ds) and all(thermal(d) for d in ds),\n      \'difference_count\':len(ds),\'differences\':[{\'path\':\'/\'.join(p),\'before\':x,\'after\':y} for p,x,y in ds]}\n\ndef main():\n    assert PILOT.is_dir() and RECOVERY.is_dir(),\'Mount Drive; original pilot and recovery directories are required.\'\n    out=ROOT/(\'scienceworld_drainage_recheck_\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\'));out.mkdir()\n    rows=[];errors=[];manifest={}\n    previous_path=ROOT/\'scienceworld_projection_smoke_20261107T132422610043Z\'/\'results.json\'\n    assert previous_path.exists(),\'Previous projection results missing; do not substitute another run.\'\n    previous={r[\'id\']:r for r in json.loads(previous_path.read_text())}\n    manifest[str(previous_path)]=hashlib.sha256(previous_path.read_bytes()).hexdigest()\n    for case in sorted(RECOVERY.glob(\'P[0-9][0-9][0-9]\')):\n        cid=case.name\n        try:\n            binding_path=case/\'action_bindings.json\'\n            bindings=json.loads(binding_path.read_text())[\'bindings\']\n            candidates={tuple(str(i) for i in b[\'obj_ids\']) for b in bindings}\n            if len(candidates)!=1:raise ValueError(\'Missing or ambiguous object-ID binding\')\n            ids=next(iter(candidates))\n            paths=list(PILOT.glob(\'task_*/\'+cid+\'/snapshots.json.gz\'));assert len(paths)==1\n            with gzip.open(paths[0],\'rt\') as f:snap=json.load(f)\n            record=json.loads((case/\'summary.json\').read_text())\n            if record[\'candidate\'].startswith(\'pour \') and len(ids)>=2 and ids[0]==ids[1]:\n                rows.append({\'id\':cid,\'status\':\'excluded_self_pour\',\'final_label\':None});continue\n            scope=make_scope(snap[\'before\'][\'tree\'],set(ids));before=project(snap[\'before\'][\'tree\'],scope)\n            row={\'id\':cid,\'candidate\':record[\'candidate\'],\'scope\':scope,\n                 \'candidate_comparison\':compare(before,project(snap[\'after\'][\'tree\'],scope)),\n                 \'recoveries\':[],\'final_label\':None,\'status\':\'diagnostic_only\'}\n            inputs=[binding_path,paths[0],case/\'summary.json\']\n            for file in sorted(case.glob(\'targeted_*.json.gz\')):\n                with gzip.open(file,\'rt\') as f:trial=json.load(f)\n                row[\'recoveries\'].append({\'undo\':trial[\'test\'][\'undo\'],\n                  \'saved_replay_stable\':trial[\'test\'][\'replay_stable\'],\n                  \'comparison\':compare(before,project(trial[\'end\'][\'tree\'],scope))})\n                inputs.append(file)\n            for file in inputs:manifest[str(file)]=hashlib.sha256(file.read_bytes()).hexdigest()\n            old=previous.get(cid, {})\n            row[\'previous_recovery_flags\']=[{\'undo\':t[\'undo\'],\n                \'exact\':t[\'comparison\'][\'exact_projection_match\'],\n                \'thermal_only\':t[\'comparison\'][\'thermal_only_residual\']} for t in old.get(\'recoveries\',[])]\n            row[\'thermal_only_flag_changed\']=any(\n                old_t[\'undo\']==new_t[\'undo\'] and old_t[\'thermal_only\']!=new_t[\'comparison\'][\'thermal_only_residual\']\n                for old_t in row[\'previous_recovery_flags\'] for new_t in row[\'recoveries\'])\n            rows.append(row)\n            print(cid,\'recovery comparisons:\',[(t[\'comparison\'][\'exact_projection_match\'],t[\'comparison\'][\'thermal_only_residual\']) for t in row[\'recoveries\']])\n        except Exception as exc:errors.append({\'id\':cid,\'error\':str(exc)})\n    tests=[t for r in rows for t in r.get(\'recoveries\',[])]\n    summary={\'old_cases_reviewed\':len(rows),\'recovery_comparisons\':len(tests),\n       \'exact_projection_matches\':sum(t[\'comparison\'][\'exact_projection_match\'] for t in tests),\n       \'thermal_only_residuals\':sum(t[\'comparison\'][\'thermal_only_residual\'] for t in tests),\n       \'excluded_self_pours\':[r[\'id\'] for r in rows if r[\'status\']==\'excluded_self_pour\'],\n       \'drainage_roots_by_case\':{r[\'id\']:r[\'scope\'][\'drainage_roots\'] for r in rows if \'scope\' in r},\n       \'cases_with_changed_thermal_only_flag\':[r[\'id\'] for r in rows if r.get(\'thermal_only_flag_changed\')],\n       \'scope_revision\':\'v2: sewer and initial contents tracked in every case; new descendants included\',\n       \'errors\':errors,\'final_labels_created\':0,\'scope_coverage_validated\':False,\n       \'partial_labels_require_exhaustive_search_for_exact_recovery\':True,\n       \'fresh_validation_complete\':False,\'saved_to\':str(out)}\n    for name,data in [(\'summary.json\',summary),(\'results.json\',rows),(\'input_hashes.json\',manifest)]:\n        (out/name).write_text(json.dumps(data,indent=2,allow_nan=False))\n    (out/\'runner.py\').write_bytes(Path(__file__).read_bytes())\n    print(\'DRAINAGE-AWARE RECHECK COMPLETE\');print(json.dumps(summary,indent=2))\n\nif __name__==\'__main__\':main()\n')
Path("/content/41_scienceworld_fresh_validation_colab.py").write_text('"""Fresh restricted-action development validation; not full-world labels."""\nimport argparse,gzip,hashlib,importlib.metadata,json,os,runpy,time,traceback\nfrom collections import deque,Counter\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nCOMMIT=\'e8216d6044e8e39be9fcb185e3b2dfb602584b52\'\nJAR=\'e77b0fee7d68abe3ca5b12e57d86e2bfe7603f20c5200da0b0f085939faeb465\'\nMANIFEST={\'version\':\'fresh_restricted_v1\',\'variations\':[1,2],\n \'task_keywords\':[\'boil\',\'freeze\',\'mix\',\'melt\'],\n \'recovery_prefixes\':[\'move \',\'activate \',\'deactivate \',\'open \',\'close \',\'pour \',\'mix \'],\n \'waiting\':[\'wait1\'], \'horizons\':[1,2,3],\n \'role_rule\':\'Initial candidate-bound objects and their descendants, plus immediate original parents; exclude sewer/universe/terminal objects. Add later descendants of original bound objects except terminals.\',\n \'selection\':\'Evenly spaced physical actions along supplied gold trajectory, before any recovery outcome is seen.\',\n \'scope\':\'v2 drainage-aware projection; initial parent subtrees additionally tracked for recovery roles\',\n \'interpretation\':\'Exhaustive only within declared action-role set. Action-count cap aborts, never truncates.\',\n \'partial\':\'Thermal-only residual with no exact witness after complete bounded search.\',\n \'read_only_actions\':\'Not members of this restricted action set.\',\n \'full_state_coverage_verified\':False}\ndef save(p,x):p.write_text(json.dumps(x,indent=2,allow_nan=False))\ndef digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,allow_nan=False).encode()).hexdigest()\ndef physical(a):return a.startswith((\'move \',\'activate \',\'deactivate \',\'pour \',\'mix \'))\ndef bindings(env,action):\n    vals={tuple(str(i) for i in r[\'obj_ids\']) for r in env.get_valid_action_object_combinations_with_templates() if r[\'action\']==action}\n    if len(vals)!=1:raise ValueError(\'Ambiguous or missing candidate binding\')\n    return set(next(iter(vals)))\ndef allowed(env,roles,roots,tree,helper,max_actions):\n    nodes,_=helper[\'index\'](tree)\n    dynamic=set(roles)|helper[\'descendants\'](nodes,roots)\n    dynamic={i for i in dynamic if i in nodes and not nodes[i].get(\'name\',\'\').lower().startswith(\'terminal \') and nodes[i].get(\'name\') not in (\'universe\',\'sewer\')}\n    grouped={}\n    for row in env.get_valid_action_object_combinations_with_templates():\n        a=row[\'action\'];ids=tuple(str(i) for i in row[\'obj_ids\'])\n        grouped.setdefault(a,set()).add(ids)\n    actions=[]\n    for a,bs in grouped.items():\n        if not a.startswith(tuple(MANIFEST[\'recovery_prefixes\'])):continue\n        eligible=[ids for ids in bs if ids and set(ids)<=dynamic and not (a.startswith(\'pour \') and len(ids)>1 and ids[0]==ids[1])]\n        if not eligible:continue\n        if len(bs)!=1:raise ValueError(\'Ambiguous recovery action: \'+a)\n        actions.append(a)\n    valid=set(env.get_valid_action_object_combinations())\n    actions.extend(a for a in MANIFEST[\'waiting\'] if a in valid)\n    actions=sorted(set(actions))\n    if len(actions)>max_actions:raise ValueError(f\'Action budget exceeded ({len(actions)}>{max_actions}); no actions silently discarded\')\n    return actions\n\ndef main(args):\n    option=\'-Xlog:os+container=off\'\n    if option not in os.environ.get(\'JAVA_TOOL_OPTIONS\',\'\').split():os.environ[\'JAVA_TOOL_OPTIONS\']=(os.environ.get(\'JAVA_TOOL_OPTIONS\',\'\')+\' \'+option).strip()\n    from scienceworld import ScienceWorldEnv\n    from scienceworld.constants import JAR_PATH\n    dist=importlib.metadata.distribution(\'scienceworld\')\n    assert json.loads(dist.read_text(\'direct_url.json\') or \'{}\').get(\'vcs_info\',{}).get(\'commit_id\')==COMMIT\n    assert hashlib.sha256(Path(JAR_PATH).read_bytes()).hexdigest()==JAR\n    helper_path=Path(\'/content/40_scienceworld_drainage_recheck_colab.py\')\n    helper=runpy.run_path(str(helper_path),run_name=\'projection_helper\')\n    root=Path(args.project);assert root.is_dir(),\'Mount Drive first.\'\n    out=root/\'results\'/(\'scienceworld_fresh_validation_\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%fZ\'));out.mkdir()\n    save(out/\'protocol.json\',{\'manifest\':MANIFEST,\'arguments\':vars(args),\'commit\':COMMIT,\'jar_sha256\':JAR,\'projection_sha256\':hashlib.sha256(helper_path.read_bytes()).hexdigest()})\n    (out/\'runner.py\').write_bytes(Path(__file__).read_bytes());(out/\'projection.py\').write_bytes(helper_path.read_bytes())\n    env=None;results=[];errors=[];started=time.monotonic()\n    try:\n        env=ScienceWorldEnv(envStepLimit=250);names=env.get_task_names();chosen=[]\n        for k in MANIFEST[\'task_keywords\']:\n            t=next((t for t in names if k in t.lower() and t not in chosen),None)\n            if t:chosen.append(t)\n        for task in chosen:\n            for variation in MANIFEST[\'variations\']:\n                folder=out/f\'{len(results):03d}_{task}_{variation}\';folder.mkdir()\n                try:\n                    print(\'Discovering:\',task,\'variation\',variation,flush=True)\n                    assert variation<env.get_max_variations(task),\'Variation unavailable\'\n                    env.load(task,variation,\'\',generateGoldPath=True);gold=env.get_gold_action_sequence()\n                    assert gold and not any(a.startswith(\'ERROR\') for a in gold),\'Gold path unavailable\'\n                    save(folder/\'gold.json\',gold)\n                    def snap(done):\n                        return {\'tree\':env.getObjectTree(),\'look\':env.look(),\'inventory\':env.inventory(),\n                            \'valid_actions\':sorted(set(env.get_valid_action_object_combinations())),\n                            \'moves\':int(env.get_num_moves()),\'score\':float(env.lastStepScore),\n                            \'goal_progress\':env.get_goal_progress(),\'done\':bool(done)}\n                    def replay(seq):\n                        env.load(task,variation,\'\',generateGoldPath=False);env.reset();done=False;trace=[]\n                        for a in seq:\n                            if done:raise ValueError(\'Terminal replay boundary\')\n                            obs,reward,done,_=env.step(a);trace.append([a,obs,reward,bool(done),int(env.get_num_moves())])\n                        return snap(done),trace\n                    env.load(task,variation,\'\',generateGoldPath=False);env.reset();prefix=[];candidates=[]\n                    for a in gold[:100]:\n                        if physical(a) and a in env.get_valid_action_object_combinations():\n                            try:\n                                ids=bindings(env,a)\n                                if not (a.startswith(\'pour \') and len(ids)<2):candidates.append((list(prefix),a,ids))\n                            except ValueError:pass\n                        _,_,done,_=env.step(a);prefix.append(a)\n                        if done:break\n                    if not candidates:raise ValueError(\'No eligible physical candidates\')\n                    if args.per_trajectory==1:indices=[len(candidates)//2]\n                    else:indices=sorted({round(i*(len(candidates)-1)/(args.per_trajectory-1)) for i in range(args.per_trajectory)})\n                    save(folder/\'selected_candidates.json\',[{\'prefix\':candidates[i][0],\'action\':candidates[i][1],\'ids\':sorted(candidates[i][2])} for i in indices])\n                    for index in indices:\n                        prefix,action,roots=candidates[index];cid=f\'V{len(results)+1:03d}\';case=folder/cid;case.mkdir()\n                        print(cid,action,flush=True)\n                        row={\'id\':cid,\'task\':task,\'variation\':variation,\'prefix\':prefix,\'candidate\':action,\'final_dataset_label\':None}\n                        nodes=[];witnesses={};queue=deque([[]]);reason=None;case_start=time.monotonic()\n                        try:\n                            before,trace=replay(prefix);again,trace2=replay(prefix)\n                            assert before==again and trace==trace2,\'Starting replay unstable\'\n                            after,at=replay(prefix+[action]);again,at2=replay(prefix+[action])\n                            assert after==again and at==at2,\'Candidate replay unstable\'\n                            all_nodes,parents=helper[\'index\'](before[\'tree\'])\n                            roles=helper[\'descendants\'](all_nodes,roots)\n                            roles|={parents[i] for i in roles if parents[i] is not None}\n                            roles={i for i in roles if all_nodes[i].get(\'name\') not in (\'universe\',\'sewer\') and not all_nodes[i].get(\'name\',\'\').startswith(\'terminal \')}\n                            scope=helper[\'make_scope\'](before[\'tree\'],roots|roles)\n                            row[\'scope\']=scope;row[\'recovery_role_ids\']=sorted(roles)\n                            reference=helper[\'project\'](before[\'tree\'],scope)\n                            with gzip.open(case/\'initial.json.gz\',\'wt\') as f:json.dump({\'before\':before,\'after\':after},f)\n                            while queue:\n                                if len(nodes)>=args.max_nodes or time.monotonic()-case_start>=args.seconds_per_case:\n                                    reason=\'budget_exhausted\';break\n                                undo=queue.popleft();end,etrace=replay(prefix+[action]+undo)\n                                comp=helper[\'compare\'](reference,helper[\'project\'](end[\'tree\'],scope))\n                                kind=\'exact\' if comp[\'exact_projection_match\'] else \'thermal\' if comp[\'thermal_only_residual\'] else None\n                                if kind and kind not in witnesses:\n                                    repeat,rtrace=replay(prefix+[action]+undo)\n                                    assert end==repeat and etrace==rtrace,\'Witness replay unstable\'\n                                    witnesses[kind]={\'undo\':undo,\'depth\':len(undo),\'independent_replay_matches\':True}\n                                    with gzip.open(case/(kind+\'_witness.json.gz\'),\'wt\') as f:json.dump({\'end\':end,\'trace\':etrace,\'comparison\':comp},f)\n                                next_actions=[]\n                                if len(undo)<3 and not end[\'done\']:\n                                    next_actions=allowed(env,roles,roots,end[\'tree\'],helper,args.max_actions)\n                                    queue.extend(undo+[a] for a in next_actions)\n                                nodes.append({\'undo\':undo,\'snapshot_sha256\':digest(end),\'kind\':kind,\'done\':end[\'done\'],\'children\':next_actions})\n                                if kind==\'exact\':reason=\'exact_witness_found\';break\n                            if reason is None:reason=\'restricted_search_exhausted\'\n                        except Exception as exc:reason=\'validation_error\';row[\'error\']=str(exc);(case/\'error.txt\').write_text(traceback.format_exc())\n                        counts=Counter(len(n[\'undo\']) for n in nodes)\n                        per_h={}\n                        for h in [1,2,3]:\n                            exact=witnesses.get(\'exact\');thermal=witnesses.get(\'thermal\')\n                            if exact and exact[\'depth\']<=h:label=\'reversible_under_projection\'\n                            elif reason==\'restricted_search_exhausted\':label=\'partially_reversible_under_projection\' if thermal and thermal[\'depth\']<=h else \'irreversible_under_action_set\'\n                            else:label=\'unresolved\'\n                            per_h[str(h)]=label\n                        row.update(status=reason,provisional_results_by_horizon=per_h,witnesses=witnesses,\n                            nodes_tested=len(nodes),nodes_by_depth=dict(counts),queued_remaining=len(queue),\n                            elapsed_seconds=round(time.monotonic()-case_start,2),scope_coverage_validated=False)\n                        save(case/\'nodes.json\',nodes);save(case/\'result.json\',row);results.append(row)\n                        save(out/\'progress.json\',results);print(\' \',reason,per_h,flush=True)\n                except Exception as exc:\n                    errors.append({\'task\':task,\'variation\':variation,\'error\':str(exc)});(folder/\'error.txt\').write_text(traceback.format_exc())\n                    print(\'Trajectory error saved:\',str(exc),flush=True)\n    finally:\n        summary={\'fresh_cases\':len(results),\'status_counts\':dict(Counter(r[\'status\'] for r in results)),\n          \'provisional_horizon3_counts\':dict(Counter(r[\'provisional_results_by_horizon\'][\'3\'] for r in results)),\n          \'errors\':errors,\'elapsed_minutes\':round((time.monotonic()-started)/60,2),\n          \'final_labels_created\':0,\'scope_coverage_validated\':False,\'scienceworld_requirement_met\':False,\'saved_to\':str(out)}\n        save(out/\'results.json\',results);save(out/\'summary.json\',summary)\n        if env:\n            try:env.close()\n            except Exception:pass\n    print(\'FRESH RESTRICTED VALIDATION COMPLETE\');print(json.dumps(summary,indent=2))\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--project\',default=\'/content/drive/MyDrive/ADNN_Irreversibility\')\n    p.add_argument(\'--per-trajectory\',type=int,default=1);p.add_argument(\'--max-nodes\',type=int,default=512)\n    p.add_argument(\'--seconds-per-case\',type=float,default=90);p.add_argument(\'--max-actions\',type=int,default=24)\n    args=p.parse_args();assert min(args.per_trajectory,args.max_nodes,args.seconds_per_case,args.max_actions)>0;main(args)\n')
print("Projection helper and fresh validation script ready.")


In [ ]:
import runpy,sys
previous_argv=sys.argv[:]
try:
    sys.argv=['/content/41_scienceworld_fresh_validation_colab.py']
    _ = runpy.run_path(sys.argv[0],run_name='__main__')
finally:
    sys.argv=previous_argv


Send the final FRESH RESTRICTED VALIDATION COMPLETE summary and any errors. Do not immediately scale the run. This pilot may reveal that exact temperature retention or the search branching prevents usable labels; that outcome must be retained.
